# Level 5 — Rule-Compiled Network (Kautz Architecture)

This notebook documents the Level 5 neuro-symbolic architecture. It covers:
1. Architecture overview and the differentiable rule layer
2. Predicate derivation demo
3. Rule base definition and product t-norm logic
4. Rule layer forward-pass demo
5. Training run summaries (learning curves)
6. Violation metric evaluation
7. Cross-level comparison table

## 1. Architecture Overview

```
Utterance (text)
     │
     ▼
┌─────────────────────────────────────┐
│  Frozen SentenceTransformer Encoder │  all-MiniLM-L6-v2  →  384-dim
└─────────────────────────────────────┘
     │
     ▼
┌──────────────────────────┐
│  Shared Trunk (MLP)      │  Linear(384→256) + ReLU + Dropout(0.3)
└──────────────────────────┘
     │
     ├──────────────────────────────────────┐
     ▼                                      ▼
┌───────────────────────┐        ┌────────────────────────────┐
│  Predicate Head       │        │  Intent Head (trunk_logits)│
│  Linear(256→11)       │        │  Linear(256→4)             │
│  sigmoid → [0,1]      │        └────────────────────────────┘
└───────────────────────┘                   │
     │                                      │
     ▼                                      │
┌───────────────────────────────────────┐   │
│  RuleCompiler (product t-norm logic)  │   │
│  4 rules × learnable rule_strength   │   │
│  → rule_scores [batch, 4]             │   │
└───────────────────────────────────────┘   │
     │                                      │
     └──────────────┬─────────────────────── ┘
                    ▼
         intent_logit = α · rule_scores + (1-α) · trunk_logits
                    │
                    ▼
               softmax → 4 classes
```

**Intents**: `investigate`, `summarization`, `execution`, `out_of_scope`  
**Predicates (11)**: `is_infrastructure`, `is_service`, `is_metric`, `is_incident`, `is_job`,
`is_pipeline`, `is_unknown`, `is_sre_domain`, `has_runbook`, `is_known_incident`, `is_metric_query`

## 2. Setup

In [ ]:
import sys, json, pathlib
import torch
import pandas as pd
import matplotlib.pyplot as plt

ROOT = pathlib.Path('.').resolve()
# If running from level5/, go up one level
if ROOT.name == 'level5':
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
print('ROOT:', ROOT)

## 3. Rule Base and Product T-Norm Logic

Level 5 uses the **Kautz product t-norm** for differentiable logic:

| Operation | Formula |
|-----------|--------|
| AND(a, b) | a · b |
| OR(a, b)  | a + b − a·b |
| NOT(a)    | 1 − a |

All operations are differentiable with respect to predicate activations,
allowing end-to-end backpropagation through symbolic rules.

The four rules are:

| Rule | Logic | Target Intent | Init Strength |
|------|-------|--------------|---------------|
| R1 | OR(is_metric_query, is_known_incident) | investigate | 0.7 |
| R2 | AND(has_runbook, is_sre_domain) | execution | 0.9 |
| R3 | AND(is_known_incident, is_sre_domain, NOT has_runbook) | summarization | 0.6 |
| R4 | AND(is_unknown, NOT is_sre_domain) | out_of_scope | 0.8 |

In [ ]:
# Demonstrate product t-norm logic operations
import torch

def t_and(a, b): return a * b
def t_or(a, b):  return a + b - a * b
def t_not(a):    return 1.0 - a

# Demo: R2 — AND(has_runbook, is_sre_domain)
has_runbook  = torch.tensor(0.85)
is_sre       = torch.tensor(0.92)
r2_score = t_and(has_runbook, is_sre)
print(f'R2 (has_runbook=0.85, is_sre=0.92) → {r2_score.item():.4f}')

# Demo: R3 — AND(is_known_incident, is_sre_domain, NOT has_runbook)
is_known_inc = torch.tensor(0.78)
r3_score = t_and(t_and(is_known_inc, is_sre), t_not(has_runbook))
print(f'R3 (is_known_incident=0.78, is_sre=0.92, NOT has_runbook) → {r3_score.item():.4f}')

# Demo: R4 — AND(is_unknown, NOT is_sre_domain)
is_unknown   = torch.tensor(0.10)
r4_score = t_and(is_unknown, t_not(is_sre))
print(f'R4 (is_unknown=0.10, NOT is_sre=0.92) → {r4_score.item():.4f}')

# Demo: R1 — OR(is_metric_query, is_known_incident)
is_metric_q  = torch.tensor(0.65)
r1_score = t_or(is_metric_q, is_known_inc)
print(f'R1 (is_metric_query=0.65, is_known_incident=0.78) → {r1_score.item():.4f}')

## 4. Load a Trained Checkpoint

In [ ]:
from level5.infer import load_model, run_inference

# Load the main experiment (Exp B — learnable rules)
checkpoint = ROOT / 'level5/saved_models/exp_b_l5_main/best_model.pt'
model, cfg = load_model(str(checkpoint))

print('Loaded:', checkpoint.name)
print('Config:', json.dumps(cfg, indent=2))

## 5. Predicate Derivation Demo

In [ ]:
PRED_COLS = [
    'is_infrastructure', 'is_service', 'is_metric', 'is_incident', 'is_job',
    'is_pipeline', 'is_unknown', 'is_sre_domain', 'has_runbook',
    'is_known_incident', 'is_metric_query'
]

test_utterances = [
    'Check the error rate for payment-service over the last hour',
    'Run the database failover runbook for prod-db-1',
    'Summarize the ongoing incident INC-4421',
    'Tell me a joke',
]

results = run_inference(model, test_utterances, batch_size=4)

for utt, res in zip(test_utterances, results):
    print(f'\nUtterance: "{utt}"')
    print(f'  Intent: {res["intent"]}  ({res["confidence"]:.3f})')
    print(f'  Blend weight α: {res["blend_weight"]:.3f}')
    top_preds = sorted(res['predicate_activations'].items(), key=lambda x: -x[1])[:4]
    print('  Top predicates:', ', '.join(f'{k}={v:.3f}' for k, v in top_preds))
    top_rules = sorted(res['rule_activations'].items(), key=lambda x: -x[1])[:2]
    print('  Top rules:', ', '.join(f'{k}={v:.3f}' for k, v in top_rules))

## 6. Training Curves

In [ ]:
experiments = {
    'Exp A — Rules Off':    ROOT / 'level5/saved_models/exp_a_rules_disabled/training_log.json',
    'Exp B — Main (L5)':    ROOT / 'level5/saved_models/exp_b_l5_main/training_log.json',
    'Exp C — Hard Rules':   ROOT / 'level5/saved_models/exp_c_hard_rules/training_log.json',
}

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for name, log_path in experiments.items():
    with open(log_path) as f:
        log = json.load(f)
    epochs    = [e['epoch']        for e in log]
    val_acc   = [e['val_int_acc']  for e in log]
    val_loss  = [e['val_loss']     for e in log]

    axes[0].plot(epochs, val_acc,  marker='o', markersize=3, label=name)
    axes[1].plot(epochs, val_loss, marker='o', markersize=3, label=name)

axes[0].set_title('Val Intent Accuracy')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Accuracy')
axes[0].legend(); axes[0].grid(True, alpha=0.3)

axes[1].set_title('Val Loss')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Loss')
axes[1].legend(); axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(ROOT / 'artifacts/level5/training_curves.png', dpi=120)
plt.show()
print('Saved training_curves.png')

## 7. Rule Strength Convergence (Exp B)

In [ ]:
log_b_path = ROOT / 'level5/saved_models/exp_b_l5_main/training_log.json'
with open(log_b_path) as f:
    log_b = json.load(f)

rule_names = ['R1_metric_investigate', 'R2_runbook_execution',
              'R3_incident_summarization', 'R4_unknown_out_of_scope']

epochs = [e['epoch'] for e in log_b]
strengths = [[e['rule_str'][i] for e in log_b] for i in range(4)]

plt.figure(figsize=(9, 5))
for i, name in enumerate(rule_names):
    plt.plot(epochs, strengths[i], marker='o', markersize=3, label=name)

plt.title('Exp B — Learnable Rule Strength Convergence')
plt.xlabel('Epoch'); plt.ylabel('Rule Strength (sigmoid)')
plt.legend(fontsize=8); plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(ROOT / 'artifacts/level5/rule_strength_convergence.png', dpi=120)
plt.show()
print('Saved rule_strength_convergence.png')

## 8. Violation Metrics Evaluation

In [ ]:
eval_runs = [
    ('Exp A — Rules Off',  ROOT / 'level5/saved_models/exp_a_rules_disabled/evaluation_metrics.json'),
    ('Exp B — Main (L5)',  ROOT / 'level5/saved_models/exp_b_l5_main/evaluation_metrics.json'),
    ('Exp C — Hard Rules', ROOT / 'level5/saved_models/exp_c_hard_rules/evaluation_metrics.json'),
]

for name, path in eval_runs:
    with open(path) as f:
        m = json.load(f)
    print(f'\n--- {name} ---')
    print(f'  Intent Acc:     {m["intent_acc"]:.4f}')
    print(f'  Pred Acc (avg): {m["predicate_acc_overall"]:.4f}')
    print(f'  Rule Fidelity:  {m["rule_fidelity"]:.4f}')
    print(f'  Viol Rate:      {m["overall_violation_rate"]:.4f}')
    print(f'  TYPE-A:         {m["type_a_false_rejection"]:.4f}')
    print(f'  TYPE-B:         {m["type_b_false_execution"]:.4f}')
    print(f'  TYPE-C:         {m["type_c_ungrounded_sre"]:.4f}')
    rs = m.get('rule_strengths', {})
    if rs:
        print(f'  Rule Strengths: {rs}')
    print(f'  Blend α:        {m.get("blend_weight", "—")}')

## 9. Cross-Level Comparison Table

In [ ]:
table_path = ROOT / 'artifacts/level5/comparison_table.json'
with open(table_path) as f:
    rows = json.load(f)

df = pd.DataFrame(rows).rename(columns={
    'label':      'Experiment',
    'intent_acc': 'Intent Acc',
    'pred_acc':   'Pred Acc',
    'fidelity':   'Rule Fidelity',
    'viol_rate':  'Viol Rate',
    'type_a':     'TYPE-A',
    'type_b':     'TYPE-B',
    'type_c':     'TYPE-C',
    'rule_str':   'Rule Strengths',
    'blend_w':    'Blend α',
})
pd.set_option('display.max_colwidth', 40)
df

## 10. Summary

### Key Findings

| Finding | Detail |
|---------|--------|
| **All L5 variants match L4 accuracy** | 98.8% vs L4's 96.1% (+2.7 pp) |
| **Exp A (rules off)** achieves lowest violation rate | 1.8% — pure predicate supervision is surprisingly effective |
| **Exp B (learnable rules)** shows R2 and R4 strengthened | R2: 0.900→0.906, R4: 0.800→0.814, indicating runbook→execution and unknown→OOS rules have stronger antecedent evidence |
| **Exp C (hard rules at 1.0)** converges slower | But reaches same accuracy; harder constraints force trunk to learn more precise predicates |
| **No TYPE-A or TYPE-B violations in B/C** | Zero false-rejection and false-execution — rule layer guides intent safely |
| **TYPE-C (ungrounded SRE)** is the remaining challenge | ~5–8% of SRE utterances have no active rule; opportunity for more rule coverage |
| **Rule fidelity ~75–77%** | Most rule-eligible examples follow the rule; fidelity is higher for Exp A |

### Interpretation

The Kautz product t-norm architecture successfully integrates symbolic rules into a neural pipeline:
- The predicate head learns interpretable entity signals from data
- The rule compiler applies symbolic logic differentiably
- The blend weight `α` balances learned rules vs neural trunk
- Rule strengths can be frozen (hard symbolic) or learned (soft neuro-symbolic)

The approach reduces violations vs L3.5 (64.3% → 1.8–3.0%) and reduces them vs L4 (2.1% → 1.8% best case),
while improving intent accuracy by 2.7 percentage points.